In [ ]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('housing_data_for_recomendation_system.csv').drop(columns={'Unnamed: 0'})

In [ ]:
df.head()

,url,image_url,beds,baths,sqft,sqft_lot,address,estimated_monthly_payment,property_type,price_per_sqft,year_built,region,interior_features,other_rooms,appliances,utilities_Electric,utilities_Sewer,utilities_Water,utilities_Utilities,parking_total_spaces,parking_parking_features,walk_score,bike_score,elementary_school_name,elementary_school_distance,middle_school_name,middle_school_distance,high_school_name,high_school_distance,flood_risk,fire_risk,wind_risk,air_risk,heat_risk,nearby_cities,property_history,parking_uncovered_spaces,parking_garage_spaces,price,sqft_lot_caterogy,sqft_lot_cleaned,zip_code,property_age,mobility_score,avg_school_distance_miles,weighted_risk_score,parking_spaces,parking_quality_score,has_garage,parking_adequacy,county,zip_price_tier
0,https://www.zillow.com/homedetails/197-Lowell-...,https://photos.zillowstatic.com/fp/90454dcdc33...,3.0,2.0,1415.0,"5,001 sqft","197 Lowell St, Arlington, MA 02474",6225.0,single family,706.0,1880.0,Arlington,fireplace; stainless steel; marble,living room; office; dining room,range; dryer; washer,circuit breakers,public sewer,public,"for gas range, for gas dryer, washer hookup",4.0,off street,81.0,61.0,Nearby,NaN,Ottoson,0.4,Arlington,1.4,1.0,1.0,6.0,3.0,5.0,Arlington; Cambridge; Everett; Framingham; Lowell,"{'date': '7/15/2025', 'event': 'Listed for sal...",NaN,NaN,999000,Ultra Compact,5.0,2474,145.0,73.0,0.900,2.85,4.0,0.0,0,Adequate,Middlesex,Mid-tier
1,https://www.zillow.com/homedetails/225-Mystic-...,https://photos.zillowstatic.com/fp/e92a0b0b33e...,2.0,1.0,1064.0,"4,818 sqft","225 Mystic St, Arlington, MA 02474",4978.0,single family,751.0,1880.0,Arlington,granite countertops; stainless steel; fireplace,bonus room; living room; dining room,range; oven,"circuit breakers, 100 amp service",public sewer,public,"for gas range, for electric oven",3.0,"paved drive, tandem, paved",69.0,59.0,Bishop,0.1,Ottoson,0.1,Arlington,0.7,1.0,1.0,6.0,3.0,5.0,Arlington; Cambridge; Everett; Framingham; Lowell,"{'date': '6/25/2025', 'event': 'Listed for sal...",1.0,NaN,799000,Ultra Compact,4.0,2474,145.0,65.0,0.300,2.85,3.0,1.0,0,Adequate,Middlesex,Mid-tier
2,https://www.zillow.com/homedetails/43-Longmead...,https://photos.zillowstatic.com/fp/740e93bfcff...,2.0,1.0,1628.0,"7,954 sqft","43 Longmeadow Rd, Arlington, MA 02474",5603.0,single family,552.0,1952.0,Arlington,fireplace; laminate,living room; family room; dining room,range; oven; washer,circuit breakers,public sewer,public,"for gas range, for gas oven, for gas dryer, wa...",5.0,off street,32.0,35.0,Nearby,5.0,Ottoson,0.2,Additional,5.0,1.0,1.0,6.0,3.0,5.0,Arlington; Cambridge; Everett; Framingham; Lowell,"{'date': '7/22/2025', 'event': 'Listed for sal...",NaN,NaN,899000,Ultra Compact,7.0,2474,73.0,33.2,3.400,2.85,5.0,0.0,0,Abundant,Middlesex,Mid-tier
3,https://www.zillow.com/homedetails/11-Pine-Ct-...,https://photos.zillowstatic.com/fp/bb84e595f96...,3.0,2.0,1824.0,"6,037 sqft","11 Pine Ct, Arlington, MA 02476",6219.0,single family,547.0,1926.0,Arlington,fireplace; hardwood floors,living room; office; dining room,range,NaN,public sewer,public,NaN,2.0,off street,84.0,54.0,Brackett,0.7,Ottoson,0.7,Arlington,0.7,1.0,1.0,6.0,3.0,6.0,Arlington; Cambridge; Everett; Framingham; Lowell,"{'date': '7/22/2025', 'event': 'Listed for sal...",NaN,NaN,998000,Ultra Compact,6.0,2476,99.0,72.0,0.700,2.95,2.0,0.0,0,Basic,Middlesex,Mid-tier
4,https://www.zillow.com/homedetails/17-Norcross...,https://photos.zillowstatic.com/fp/ac474eb496d...,2.0,2.0,1221.0,NaN,"17 Norcross St FLOOR 3, Arlington, MA 02474",4339.0,condo,547.0,1908.0,Arlington,fireplace; hardwood floor,living room; family room; dining room,range,circuit breakers,public sewer,public,NaN,1.0,"off street, tandem, paved",77.0,96.0,Thompson,0.1,Ottoson,0.1,Major,5.0,1.0,1.0,6.0,3.0,6.0,Arlington; Cambridge; Everett; Framingham; Lowell,"{'date': '6/21/2025', 'event': 'Price change',...",1.0,NaN,668000,Unknown,NaN,2474,117.0,84.6,1.733,2.95,1.0,1.0,0,Limited,Middlesex,Mid-tier


In [ ]:
df.shape

(7941, 52)

# Columns to use

address, interior_features	other_rooms	appliances	utilities_Electric	utilities_Sewer	utilities_Water	utilities_Utilitie sparking_parking_features

`nearby_cities`

In [ ]:
df['utilities_Water'].unique()

array(['public', 'private', 'public, private', 'individual meter', nan,
       'well', 'private, shared well', 'buzzards bay', 'shared well',
       'other', 'municipal', 'private, other', 'private, well', 'none',
       'public, private, other', 'atlantic ocean', 'in fee',
       'municipal, public', 'great pond', 'town', 'nantucket sound',
       'pickerel pond', 'hyannis port harbor', 'provincetown harbor',
       'individual meter, municipal, public', 'bass river',
       'public, other', 'public, private, cistern',
       'public, individual meter', 'public, well',
       "well, town water available at street; sewer: private septic; parcel id: 112-10-0; zoning: r1.   desirable area close to highly-rated public &amp; private schools, conservation &amp; cultural attractions. 3+/- mi to walden pond &amp; minuteman national historic park hq, 4+/- mi to decordova &amp; flint's pond, 5+/- mi to lincoln center, 6+/- miles to ma audubon's drumlin farm.  commuter convenience with easy acce

In [ ]:
df[df['utilities_Water'] == 'currently on city water, with a well available for plumbing and softener system options. adu potential: build an accessory dwelling unit to maximize space and value. prime location: minutes to the beach, marinas, and all the best cape cod amenities. stunning views: enjoy unobstructed vistas of the sagamore bridge and cape cod canal from multiple rooms. this home offers incredible potential to create your dream cape escape. with summer around the corner, now is the perfect time to secure this one-of-a-kind property.']

,url,image_url,beds,baths,sqft,sqft_lot,address,estimated_monthly_payment,property_type,price_per_sqft,year_built,region,interior_features,other_rooms,appliances,utilities_Electric,utilities_Sewer,utilities_Water,utilities_Utilities,parking_total_spaces,parking_parking_features,walk_score,bike_score,elementary_school_name,elementary_school_distance,middle_school_name,middle_school_distance,high_school_name,high_school_distance,flood_risk,fire_risk,wind_risk,air_risk,heat_risk,nearby_cities,property_history,parking_uncovered_spaces,parking_garage_spaces,price,sqft_lot_caterogy,sqft_lot_cleaned,zip_code,property_age,mobility_score,avg_school_distance_miles,weighted_risk_score,parking_spaces,parking_quality_score,has_garage,parking_adequacy,county,zip_price_tier
2884,https://www.zillow.com/homedetails/6-Brigantin...,https://photos.zillowstatic.com/fp/6c07a3897a0...,3.0,3.0,3144.0,1.01 Acres,"6 Brigantine Passage Drive, Buzzards Bay, MA 0...",5682.0,single family,286.0,1994.0,Bourne,fireplace; carpet; skylight; laminate,office; dining room; living room,range; dishwasher; refrigerator,NaN,septic tank,"currently on city water, with a well available...",NaN,12.0,basement,16.0,49.0,Bournedale,2.0,Bourne,2.0,Bourne,3.5,1.0,1.0,8.0,1.0,5.0,Barnstable; Bourne; Brewster; Chatham; Dennis,"{'date': '6/9/2025', 'event': 'Price change', ...",1.0,2.0,899900,Rural/Estate,43995.6,2532,31.0,29.2,2.5,3.15,12.0,4.0,1,Abundant,Barnstable,Mid-tier


In [ ]:
df.loc[2884, 'utilities_Water'] = 'None'


In [ ]:
df.iloc[2884]['utilities_Water']

'None'

In [ ]:
df[df['utilities_Water'] == "well, town water available at street; sewer: private septic; parcel id: 112-10-0; zoning: r1.   desirable area close to highly-rated public &amp; private schools, conservation &amp; cultural attractions. 3+/- mi to walden pond &amp; minuteman national historic park hq, 4+/- mi to decordova &amp; flint's pond, 5+/- mi to lincoln center, 6+/- miles to ma audubon's drumlin farm.  commuter convenience with easy access to mbta commuter rail, routes 2, 2a, 117, 128, i-95 &amp; i-495."]

,url,image_url,beds,baths,sqft,sqft_lot,address,estimated_monthly_payment,property_type,price_per_sqft,year_built,region,interior_features,other_rooms,appliances,utilities_Electric,utilities_Sewer,utilities_Water,utilities_Utilities,parking_total_spaces,parking_parking_features,walk_score,bike_score,elementary_school_name,elementary_school_distance,middle_school_name,middle_school_distance,high_school_name,high_school_distance,flood_risk,fire_risk,wind_risk,air_risk,heat_risk,nearby_cities,property_history,parking_uncovered_spaces,parking_garage_spaces,price,sqft_lot_caterogy,sqft_lot_cleaned,zip_code,property_age,mobility_score,avg_school_distance_miles,weighted_risk_score,parking_spaces,parking_quality_score,has_garage,parking_adequacy,county,zip_price_tier,"(2884, utilities_Water)"
1306,https://www.zillow.com/homedetails/6-Emerson-R...,https://photos.zillowstatic.com/fp/b97ff57c14c...,3.0,2.0,2245.0,2.8 Acres,"6 Emerson Rd, Lincoln, MA 01773",4801.0,single family,410.0,1978.0,Lincoln,fireplace,den; basement; office,range,amps(0),private septic; parcel id: 112-10-0; zoning: r...,"well, town water available at street; sewer: p...",NaN,NaN,driveway,4.0,13.0,high,1.5,Homes in,5.0,Desirable area close to,5.0,1.0,2.0,6.0,2.0,4.0,Arlington; Cambridge; Everett; Framingham; Lowell,NaN,1.0,NaN,989700,Rural/Estate,121968.0,1773,47.0,7.6,3.833,2.8,1.0,1.0,0,Limited,Middlesex,High-end,None


In [ ]:
df.loc[1306, 'utilities_Water'] = 'None'


In [ ]:
df[df['utilities_Water'] == "set up your beach chair at low tide or float on a raft at high tide in your private cove and watch the river and boats flow by"]

,url,image_url,beds,baths,sqft,sqft_lot,address,estimated_monthly_payment,property_type,price_per_sqft,year_built,region,interior_features,other_rooms,appliances,utilities_Electric,utilities_Sewer,utilities_Water,utilities_Utilities,parking_total_spaces,parking_parking_features,walk_score,bike_score,elementary_school_name,elementary_school_distance,middle_school_name,middle_school_distance,high_school_name,high_school_distance,flood_risk,fire_risk,wind_risk,air_risk,heat_risk,nearby_cities,property_history,parking_uncovered_spaces,parking_garage_spaces,price,sqft_lot_caterogy,sqft_lot_cleaned,zip_code,property_age,mobility_score,avg_school_distance_miles,weighted_risk_score,parking_spaces,parking_quality_score,has_garage,parking_adequacy,county,zip_price_tier,"(2884, utilities_Water)"
6285,https://www.zillow.com/homedetails/7-Cove-Way-...,https://photos.zillowstatic.com/fp/f1a8ea0a78e...,4.0,2.0,2499.0,0.84 Acres,"7 Cove Way, Gloucester, MA 01930",16122.0,single family,1038.0,1920.0,Gloucester,carpet; skylight; marble; fireplace; stainless...,office; den; living room,washer; range; oven,200+ amp service,private sewer,set up your beach chair at low tide or float o...,"for gas oven, for gas dryer, washer hookup",9.0,"detached, garage door opener, storage, garage ...",19.0,7.0,Nearby,NaN,maley,1.1,Nearby,2.0,1.0,1.0,8.0,2.0,5.0,Andover; Beverly; Haverhill; Lawrence; Lynn,"{'date': '6/2/2025', 'event': 'Listed for sale...",1.0,3.0,2595000,Suburban Large,36590.4,1930,105.0,14.2,1.55,3.25,9.0,5.5,1,Abundant,Essex,Mid-tier,None


In [ ]:
df.loc[6285, 'utilities_Water'] = 'None'

In [ ]:
df['utilities_Water'].unique()

array(['public', 'private', 'public, private', 'individual meter', nan,
       'well', 'private, shared well', 'buzzards bay', 'shared well',
       'other', 'municipal', 'private, other', 'private, well', 'none',
       'public, private, other', 'atlantic ocean', 'in fee',
       'municipal, public', 'great pond', 'town', 'nantucket sound',
       'pickerel pond', 'hyannis port harbor', 'provincetown harbor',
       'individual meter, municipal, public', 'bass river',
       'public, other', 'public, private, cistern',
       'public, individual meter', 'public, well', 'None',
       'well, private', 'assessment seller, in fee', 'quanset pond',
       'crocker pond', 'santuit pond', "stewart's creek",
       'wequaquet lake', 'cataquin pond', 'jabinettes pond',
       'seymour pond', 'pleasant bay', 'bay view beach', 'lewis pond',
       'sheep pond', 'bucks creek marshes', 'buttermilk bay',
       'cape cod bay', 'grand cove', 'hyannis harbor', 'widger hole',
       'pochet inlet', '

In [ ]:
df.loc[4532, 'utilities_Water']

'public'

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))

In [ ]:
df['utilities_Water'] = df['utilities_Water'].fillna('')
tfidf_matrix = tfidf_vectorizer.fit_transform(df['utilities_Water'])

In [ ]:
tfidf_matrix.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
cosine_sim1 = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [ ]:
cosine_sim1.shape

(7941, 7941)

In [ ]:
def recommend_properties_by_water_utility(water_utility_description, tfidf_vectorizer=tfidf_vectorizer, tfidf_matrix=tfidf_matrix, df=df):
    # Vectorize the input water utility description
    input_vector = tfidf_vectorizer.transform([water_utility_description])

    # Calculate cosine similarity between the input vector and the existing property vectors
    sim_scores = cosine_similarity(input_vector, tfidf_matrix).flatten()

    # Get the indices of the most similar properties
    # Use argpartition for efficiency to get the top 6 indices without full sort
    top_indices = sim_scores.argpartition(-6)[-6:]

    # Sort these top indices by their similarity scores in descending order
    sorted_indices = top_indices[np.argsort(-sim_scores[top_indices])][:]

    # Get the scores of the 5 most similar properties
    sim_scores_sorted = sim_scores[sorted_indices]

    recommendations_df = pd.DataFrame({
        'address': df['address'].iloc[sorted_indices],
        'utilities_Water': df['utilities_Water'].iloc[sorted_indices],
        'SimilarityScore': sim_scores_sorted
    })

    # Return the top 5 most similar properties
    # Filter out the exact match if it's present and has a score of 1.0, unless there are fewer than 5 results
    if len(recommendations_df) > 5 and recommendations_df.iloc[0]['SimilarityScore'] == 1.0 and recommendations_df.iloc[0]['utilities_Water'].lower() == water_utility_description.lower():
        return recommendations_df.iloc[1:6].reset_index(drop=True)
    else:
        return recommendations_df.head(5).reset_index(drop=True)

In [ ]:
recommendations = recommend_properties_by_water_utility("indian")
display(recommendations)

,address,utilities_Water,SimilarityScore
0,"454 Seneca Dr, Becket, MA 01223",indian lake,0.619773
1,"113 Iroquois Ave, Becket, MA 01223",indian lakes,0.560175
2,"360 W 2nd St UNIT 11, South Boston, MA 02127",public,0.000000
3,"1472 Commonwealth Ave #A, Brighton, MA 02135",public,0.000000
4,"80 Broad St, Boston, MA 02110",,0.000000


# Instead, i will not combine all utilities cols

**Reasoning**:
Examine the columns of the DataFrame to identify utility-related columns.



In [ ]:
utility_columns = [col for col in df.columns if isinstance(col, str) and 'utilities' in col.lower()]
print("Utility columns:", utility_columns)

Utility columns: ['utilities_Electric', 'utilities_Sewer', 'utilities_Water', 'utilities_Utilities']


In [ ]:
utility_columns = ['utilities_Electric', 'utilities_Sewer', 'utilities_Water', 'utilities_Utilities']
df['combined_utilities'] = ''
for col in utility_columns:
    df[col] = df[col].fillna('').astype(str)
    df['combined_utilities'] = df['combined_utilities'] + df[col] + ' '

df['combined_utilities'] = df['combined_utilities'].str.strip()
display(df[['address', 'utilities_Electric', 'utilities_Sewer', 'utilities_Water', 'utilities_Utilities', 'combined_utilities']].head())

,address,utilities_Electric,utilities_Sewer,utilities_Water,utilities_Utilities,combined_utilities
0,"197 Lowell St, Arlington, MA 02474",circuit breakers,public sewer,public,"for gas range, for gas dryer, washer hookup",circuit breakers public sewer public for gas r...
1,"225 Mystic St, Arlington, MA 02474","circuit breakers, 100 amp service",public sewer,public,"for gas range, for electric oven","circuit breakers, 100 amp service public sewer..."
2,"43 Longmeadow Rd, Arlington, MA 02474",circuit breakers,public sewer,public,"for gas range, for gas oven, for gas dryer, wa...",circuit breakers public sewer public for gas r...
3,"11 Pine Ct, Arlington, MA 02476",,public sewer,public,,public sewer public
4,"17 Norcross St FLOOR 3, Arlington, MA 02474",circuit breakers,public sewer,public,,circuit breakers public sewer public


## Tf-idf vectorization


In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform(df['combined_utilities'])

In [ ]:
cosine_sim_combined_utilities = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(cosine_sim_combined_utilities.shape)

(7941, 7941)


In [ ]:
def recommend_properties_by_utility_features(input_features, num_recommendations=5):
    """
    Recommends properties based on the similarity of their combined utility features.

    Args:
        input_features: An integer index of a property or a string of utility features.
        num_recommendations: The number of recommendations to return.

    Returns:
        A pandas DataFrame containing the recommended property addresses, their combined
        utility features, and their similarity scores.
    """
    if isinstance(input_features, int):
        # Get the vector for the specified property index
        input_vector = tfidf_matrix[input_features]
        # Calculate similarity scores
        sim_scores = cosine_similarity(input_vector, tfidf_matrix).flatten()
        # Exclude the input property itself
        property_indices = sim_scores.argsort()[-num_recommendations-1:][::-1]
        # Filter out the input index if it's the top result (similarity score 1.0)
        if property_indices[0] == input_features:
            property_indices = property_indices[1:]
        else:
            property_indices = property_indices[:num_recommendations]

    elif isinstance(input_features, str):
        # Vectorize the input utility features string
        input_vector = tfidf_vectorizer.transform([input_features])
        # Calculate similarity scores
        sim_scores = cosine_similarity(input_vector, tfidf_matrix).flatten()
        # Get the indices of the most similar properties
        property_indices = sim_scores.argsort()[-num_recommendations:][::-1]

    else:
        raise ValueError("Input must be an integer index or a string of utility features.")

    # Get the similarity scores for the recommended properties
    sim_scores_sorted = sim_scores[property_indices]

    # Create a DataFrame of recommendations
    recommendations_df = pd.DataFrame({
        'address': df['address'].iloc[property_indices],
        'combined_utilities': df['combined_utilities'].iloc[property_indices],
        'SimilarityScore': sim_scores_sorted,
        'Price': df['price'].iloc[property_indices]
    })

    return recommendations_df.reset_index(drop=True)


In [ ]:
recommendations_by_string = recommend_properties_by_utility_features("indian")
display(recommendations_by_string)

,address,combined_utilities,SimilarityScore,Price
0,"454 Seneca Dr, Becket, MA 01223",200 amp private sewer indian lake,0.482250,550000
1,"113 Iroquois Ave, Becket, MA 01223",200 amp private sewer indian lakes fiber optic...,0.355512,599000
2,"80 Broad St, Boston, MA 02110",,0.000000,100000
3,"21 Chestnut St, Boston, MA 02129",public sewer public,0.000000,11800000
4,"1472 Commonwealth Ave #A, Brighton, MA 02135",public sewer public for electric range,0.000000,579900


# Task
Implement a property recommendation system based on the similarity of 'utilities_water', 'interior_features', 'other_rooms', 'appliances', and 'nearby_cities' columns in the dataframe. Create separate recommendation functions for 'utilities_water', the combined 'interior_features', 'other_rooms', and 'appliances', and 'nearby_cities'. Use TF-IDF vectorization and cosine similarity for calculating similarity. Ensure the code is well-structured with minimal comments, focusing on major sections.

## Combine interior, other rooms, and appliances features

### Subtask:
Create a new column named `combined_interior_features` by concatenating the text from the 'interior_features', 'other_rooms', and 'appliances' columns. Handle any missing values by replacing them with empty strings before concatenation.


**Reasoning**:
Create a new column by concatenating the text from the specified columns, handling missing values.



In [ ]:
feature_columns = ['interior_features', 'other_rooms', 'appliances']
df['combined_interior_features'] = ''
for col in feature_columns:
    df[col] = df[col].fillna('').astype(str)
    df['combined_interior_features'] = df['combined_interior_features'] + df[col] + ' '

df['combined_interior_features'] = df['combined_interior_features'].str.strip()
display(df[feature_columns + ['combined_interior_features']].head())

,interior_features,other_rooms,appliances,combined_interior_features
0,fireplace; stainless steel; marble,living room; office; dining room,range; dryer; washer,fireplace; stainless steel; marble living room...
1,granite countertops; stainless steel; fireplace,bonus room; living room; dining room,range; oven,granite countertops; stainless steel; fireplac...
2,fireplace; laminate,living room; family room; dining room,range; oven; washer,fireplace; laminate living room; family room; ...
3,fireplace; hardwood floors,living room; office; dining room,range,fireplace; hardwood floors living room; office...
4,fireplace; hardwood floor,living room; family room; dining room,range,fireplace; hardwood floor living room; family ...


## Tf-idf vectorization for interior/other rooms/appliances

### Subtask:
Apply TF-IDF vectorization to the combined interior, other rooms, and appliances features stored in the `combined_interior_features` column.


**Reasoning**:
Apply TF-IDF vectorization to the combined interior, other rooms, and appliances features.



In [ ]:
tfidf_vectorizer_interior = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix_interior = tfidf_vectorizer_interior.fit_transform(df['combined_interior_features'])

In [ ]:
cosine_sim_interior = cosine_similarity(tfidf_matrix_interior, tfidf_matrix_interior)
print(cosine_sim_interior.shape)

(7941, 7941)


## Tf-idf vectorization for nearby cities

### Subtask:
Apply TF-IDF vectorization specifically to the 'nearby_cities' column.


In [ ]:
tfidf_vectorizer_cities = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
df['nearby_cities'] = df['nearby_cities'].fillna('')
tfidf_matrix_cities = tfidf_vectorizer_cities.fit_transform(df['nearby_cities'])

In [ ]:
cosine_sim_cities = cosine_similarity(tfidf_matrix_cities, tfidf_matrix_cities)
print(cosine_sim_cities.shape)

(7941, 7941)


In [ ]:
def recommend_properties_by_interior_features(input_features, num_recommendations=5):
    """
    Recommends properties based on the similarity of their combined interior,
    other rooms, and appliances features.

    Args:
        input_features: An integer index of a property or a string of features.
        num_recommendations: The number of recommendations to return.

    Returns:
        A pandas DataFrame containing the recommended property addresses, their combined
        interior features, their similarity scores, and price.
    """
    if isinstance(input_features, int):
        # Get the similarity scores for the specified property index
        sim_scores = cosine_sim_interior[input_features]
        # Get the indices of the most similar properties, excluding the input property itself
        property_indices = sim_scores.argsort()[-num_recommendations-1:][::-1]
        # Filter out the input index if it's the top result (similarity score 1.0)
        if property_indices[0] == input_features:
            property_indices = property_indices[1:]
        else:
            property_indices = property_indices[:num_recommendations]

    elif isinstance(input_features, str):
        # Vectorize the input features string
        input_vector = tfidf_vectorizer_interior.transform([input_features])
        # Calculate similarity scores
        sim_scores = cosine_similarity(input_vector, tfidf_matrix_interior).flatten()
        # Get the indices of the most similar properties
        property_indices = sim_scores.argsort()[-num_recommendations:][::-1]

    else:
        raise ValueError("Input must be an integer index or a string of interior features.")

    # Get the similarity scores for the recommended properties
    sim_scores_sorted = sim_scores[property_indices]

    # Create a DataFrame of recommendations
    recommendations_df = pd.DataFrame({
        'address': df['address'].iloc[property_indices],
        'combined_interior_features': df['combined_interior_features'].iloc[property_indices],
        'SimilarityScore': sim_scores_sorted,
        'Price': df['price'].iloc[property_indices]
    })

    return recommendations_df.reset_index(drop=True)

In [ ]:
recommendations_by_interior = recommend_properties_by_interior_features("fireplace; hardwood floors")
display(recommendations_by_interior)

,address,combined_interior_features,SimilarityScore,Price
0,"20 Rowes Wharf APT 309, Boston, MA 02110",fireplace; hardwood floors den; office; living...,0.586953,4250000
1,"6 Whittier Pl APT 7E, Boston, MA 02114",fireplace; hardwood floors den; office; living...,0.586953,785000
2,"37 Beacon St UNIT 24, Boston, MA 02108",fireplace; hardwood floors den; office; living...,0.586953,950000
3,"24 Wands St, Springfield, MA 01118",fireplace; hardwood floors den; office; living...,0.586953,224900
4,"26 Chiswick Rd APT 6, Brighton, MA 02135",fireplace; hardwood floors den; office; living...,0.586953,370000


In [ ]:
def recommend_properties_by_nearby_cities(input_features, num_recommendations=5):
    """
    Recommends properties based on the similarity of their 'nearby_cities' feature.

    Args:
        input_features: An integer index of a property or a string of cities.
        num_recommendations: The number of recommendations to return.

    Returns:
        A pandas DataFrame containing the recommended property addresses, their
        'nearby_cities' values, their similarity scores, and price.
    """
    if isinstance(input_features, int):
        # Get the similarity scores for the specified property index
        sim_scores = cosine_sim_cities[input_features]
        # Get the indices of the most similar properties, excluding the input property itself
        property_indices = sim_scores.argsort()[-num_recommendations-1:][::-1]
        # Filter out the input index if it's the top result (similarity score 1.0)
        if property_indices[0] == input_features:
            property_indices = property_indices[1:]
        else:
            property_indices = property_indices[:num_recommendations]

    elif isinstance(input_features, str):
        # Vectorize the input cities string
        input_vector = tfidf_vectorizer_cities.transform([input_features])
        # Calculate similarity scores
        sim_scores = cosine_similarity(input_vector, tfidf_matrix_cities).flatten()
        # Get the indices of the most similar properties
        property_indices = sim_scores.argsort()[-num_recommendations:][::-1]

    else:
        raise ValueError("Input must be an integer index or a string of nearby cities.")

    # Get the similarity scores for the recommended properties
    sim_scores_sorted = sim_scores[property_indices]

    # Create a DataFrame of recommendations
    recommendations_df = pd.DataFrame({
        'address': df['address'].iloc[property_indices],
        'nearby_cities': df['nearby_cities'].iloc[property_indices],
        'SimilarityScore': sim_scores_sorted,
        'Price': df['price'].iloc[property_indices]
    })

    return recommendations_df.reset_index(drop=True)

In [ ]:
recommendations_by_cities_string = recommend_properties_by_nearby_cities("Boston; Cambridge")
display(recommendations_by_cities_string)

,address,nearby_cities,SimilarityScore,Price
0,"36 Hall St #1, Boston, MA 02130",Boston; Chelsea; Revere; Winthrop,0.285957,1225000
1,"32 Hale Ave, Winthrop, MA 02152",Boston; Chelsea; Revere; Winthrop,0.285957,849000
2,"90 Wareham St UNIT 303, Boston, MA 02118",Boston; Chelsea; Revere; Winthrop,0.285957,699000
3,"65 E India Row #28EF, Boston, MA 02110",Boston; Chelsea; Revere; Winthrop,0.285957,2900000
4,"150 Seaport Blvd #GPH, Boston, MA 02210",Boston; Chelsea; Revere; Winthrop,0.285957,49500000


In [ ]:
recommendations_by_cities_index = recommend_properties_by_nearby_cities(0)
display(recommendations_by_cities_index)

,address,nearby_cities,SimilarityScore,Price
0,"200 Market St APT 607, Lowell, MA 01852",Arlington; Cambridge; Everett; Framingham; Lowell,1.0,375000
1,"31 Rogers Ave, Somerville, MA 02144",Arlington; Cambridge; Everett; Framingham; Lowell,1.0,1700000
2,"44 Dean Rd, Wayland, MA 01778",Arlington; Cambridge; Everett; Framingham; Lowell,1.0,879999
3,"1 Fenner St, Billerica, MA 01821",Arlington; Cambridge; Everett; Framingham; Lowell,1.0,779900
4,"10 Simonne Ln, Pepperell, MA 01463",Arlington; Cambridge; Everett; Framingham; Lowell,1.0,499900


In [ ]:
recommendations_by_interior_string = recommend_properties_by_interior_features("fireplace, hardwood floors")
display(recommendations_by_interior_string)

recommendations_by_interior_index = recommend_properties_by_interior_features(0)
display(recommendations_by_interior_index)

recommendations_by_cities_string = recommend_properties_by_nearby_cities("Boston, Cambridge")
display(recommendations_by_cities_string)

recommendations_by_cities_index = recommend_properties_by_nearby_cities(0)
display(recommendations_by_cities_index)

,address,combined_interior_features,SimilarityScore,Price
0,"20 Rowes Wharf APT 309, Boston, MA 02110",fireplace; hardwood floors den; office; living...,0.586953,4250000
1,"6 Whittier Pl APT 7E, Boston, MA 02114",fireplace; hardwood floors den; office; living...,0.586953,785000
2,"37 Beacon St UNIT 24, Boston, MA 02108",fireplace; hardwood floors den; office; living...,0.586953,950000
3,"24 Wands St, Springfield, MA 01118",fireplace; hardwood floors den; office; living...,0.586953,224900
4,"26 Chiswick Rd APT 6, Brighton, MA 02135",fireplace; hardwood floors den; office; living...,0.586953,370000


,address,combined_interior_features,SimilarityScore,Price
0,"295 Atlantic St, Quincy, MA 02171",fireplace; stainless steel; marble living room...,0.749892,729000
1,"272-272 Cross St #2B, Malden, MA 02148",fireplace; stainless steel; hardwood floors; m...,0.726840,240370
2,"94 Circuit Rd, Winthrop, MA 02152",fireplace; stainless steel; marble; hardwood f...,0.703758,899000
3,"90 Quincy Shore Dr APT 110, Quincy, MA 02171",fireplace; laminate; stainless steel; marble l...,0.672616,375000
4,"228 Auburn St #228, Auburndale, MA 02466",fireplace; marble living room; office; dining ...,0.651323,2098000


,address,nearby_cities,SimilarityScore,Price
0,"36 Hall St #1, Boston, MA 02130",Boston; Chelsea; Revere; Winthrop,0.285957,1225000
1,"32 Hale Ave, Winthrop, MA 02152",Boston; Chelsea; Revere; Winthrop,0.285957,849000
2,"90 Wareham St UNIT 303, Boston, MA 02118",Boston; Chelsea; Revere; Winthrop,0.285957,699000
3,"65 E India Row #28EF, Boston, MA 02110",Boston; Chelsea; Revere; Winthrop,0.285957,2900000
4,"150 Seaport Blvd #GPH, Boston, MA 02210",Boston; Chelsea; Revere; Winthrop,0.285957,49500000


,address,nearby_cities,SimilarityScore,Price
0,"200 Market St APT 607, Lowell, MA 01852",Arlington; Cambridge; Everett; Framingham; Lowell,1.0,375000
1,"31 Rogers Ave, Somerville, MA 02144",Arlington; Cambridge; Everett; Framingham; Lowell,1.0,1700000
2,"44 Dean Rd, Wayland, MA 01778",Arlington; Cambridge; Everett; Framingham; Lowell,1.0,879999
3,"1 Fenner St, Billerica, MA 01821",Arlington; Cambridge; Everett; Framingham; Lowell,1.0,779900
4,"10 Simonne Ln, Pepperell, MA 01463",Arlington; Cambridge; Everett; Framingham; Lowell,1.0,499900


In [ ]:
def recommend_properties_with_scores(property_input, top_n=5, weight_utilities=30, weight_interior=20, weight_cities=8):

    # Handle string input representing desired features
    if isinstance(property_input, str) and property_input not in df['address'].values:
        # Vectorize the input string for each category
        input_vector_utilities = tfidf_vectorizer.transform([property_input])
        input_vector_interior = tfidf_vectorizer_interior.transform([property_input])
        input_vector_cities = tfidf_vectorizer_cities.transform([property_input])

        # Calculate similarity scores for each category
        sim_scores_utilities = cosine_similarity(input_vector_utilities, tfidf_matrix).flatten()
        sim_scores_interior = cosine_similarity(input_vector_interior, tfidf_matrix_interior).flatten()
        sim_scores_cities = cosine_similarity(input_vector_cities, tfidf_matrix_cities).flatten()

        # Combine the similarity scores with the specified weights
        combined_sim_scores = (weight_utilities * sim_scores_utilities +
                               weight_interior * sim_scores_interior +
                               weight_cities * sim_scores_cities)

        # Get the indices of the top_n most similar properties
        # Ensure we don't go out of bounds if top_n is larger than the number of available properties
        top_indices = combined_sim_scores.argsort()[-top_n:][::-1]
        top_scores = combined_sim_scores[top_indices]

        # Create a dataframe with the results
        recommendations_df = pd.DataFrame({
            'address': df['address'].iloc[top_indices],
            'CombinedSimilarityScore': top_scores
        })

        return recommendations_df.reset_index(drop=True)

    # Handle string input representing an existing property address
    elif isinstance(property_input, str) and property_input in df['address'].values:
        idx = df.index[df['address'] == property_input].tolist()[0]

        # Combine the cosine similarity matrices with the specified weights
        combined_cosine_sim_matrix_local = (weight_utilities * cosine_sim1 +
                                      weight_interior * cosine_sim_interior +
                                      weight_cities * cosine_sim_cities)

        # Get the similarity scores for the property using its index
        sim_scores = list(enumerate(combined_cosine_sim_matrix_local[idx]))

        # Sort properties based on the similarity scores
        sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

        # Get the indices and scores of the top_n most similar properties (excluding the input property)
        # Ensure we don't go out of bounds if top_n is larger than the number of available properties
        end_index = min(top_n + 1, len(sorted_scores))
        top_indices = [i[0] for i in sorted_scores[1:end_index]]
        top_scores = [i[1] for i in sorted_scores[1:end_index]]

        # Create a dataframe with the results
        recommendations_df = pd.DataFrame({
            'address': df['address'].iloc[top_indices],
            'CombinedSimilarityScore': top_scores
        })

        return recommendations_df.reset_index(drop=True)

    # Handle integer input representing a property index
    elif isinstance(property_input, int):
        idx = property_input
        # Combine the cosine similarity matrices with the specified weights
        combined_cosine_sim_matrix_local = (weight_utilities * cosine_sim1 +
                                      weight_interior * cosine_sim_interior +
                                      weight_cities * cosine_sim_cities)

        # Get the similarity scores for the property using its index
        sim_scores = list(enumerate(combined_cosine_sim_matrix_local[idx]))

        # Sort properties based on the similarity scores
        sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

        # Get the indices and scores of the top_n most similar properties (excluding the input property)
        # Ensure we don't go out of bounds if top_n is larger than the number of available properties
        end_index = min(top_n + 1, len(sorted_scores))
        top_indices = [i[0] for i in sorted_scores[1:end_index]]
        top_scores = [i[1] for i in sorted_scores[1:end_index]]

        # Create a dataframe with the results
        recommendations_df = pd.DataFrame({
            'address': df['address'].iloc[top_indices],
            'Price': df['price'].iloc[top_indices],
            'CombinedSimilarityScore': top_scores
        })

        return recommendations_df.reset_index(drop=True)

    else:
        return "Invalid input type. Please provide a property address (string), index (integer), or a string of desired features."

# Test the recommender function with a string of desired features
desired_features_string = "walk-in closet; marble; fireplace detached private"
recommendations_by_features = recommend_properties_with_scores(desired_features_string)
display(recommendations_by_features)

,address,CombinedSimilarityScore
0,"28 Picket Fence, Plymouth, MA 02360",31.810231
1,"130 Forest Ave, Cohasset, MA 02025",31.359014
2,"400 Rathbun Rd, Hancock, MA 01237",31.054805
3,"40 Firefly Point #40, Plymouth, MA 02360",30.355484
4,"13 Johnson Way, Rutland, MA 01543",30.293332


In [ ]:
df.head()

,url,image_url,beds,baths,sqft,sqft_lot,address,estimated_monthly_payment,property_type,price_per_sqft,year_built,region,interior_features,other_rooms,appliances,utilities_Electric,utilities_Sewer,utilities_Water,utilities_Utilities,parking_total_spaces,parking_parking_features,walk_score,bike_score,elementary_school_name,elementary_school_distance,middle_school_name,middle_school_distance,high_school_name,high_school_distance,flood_risk,fire_risk,wind_risk,air_risk,heat_risk,nearby_cities,property_history,parking_uncovered_spaces,parking_garage_spaces,price,sqft_lot_caterogy,sqft_lot_cleaned,zip_code,property_age,mobility_score,avg_school_distance_miles,weighted_risk_score,parking_spaces,parking_quality_score,has_garage,parking_adequacy,county,zip_price_tier,"(2884, utilities_Water)",combined_utilities,combined_interior_features
0,https://www.zillow.com/homedetails/197-Lowell-...,https://photos.zillowstatic.com/fp/90454dcdc33...,3.0,2.0,1415.0,"5,001 sqft","197 Lowell St, Arlington, MA 02474",6225.0,single family,706.0,1880.0,Arlington,fireplace; stainless steel; marble,living room; office; dining room,range; dryer; washer,circuit breakers,public sewer,public,"for gas range, for gas dryer, washer hookup",4.0,off street,81.0,61.0,Nearby,NaN,Ottoson,0.4,Arlington,1.4,1.0,1.0,6.0,3.0,5.0,Arlington; Cambridge; Everett; Framingham; Lowell,"{'date': '7/15/2025', 'event': 'Listed for sal...",NaN,NaN,999000,Ultra Compact,5.0,2474,145.0,73.0,0.900,2.85,4.0,0.0,0,Adequate,Middlesex,Mid-tier,None,circuit breakers public sewer public for gas r...,fireplace; stainless steel; marble living room...
1,https://www.zillow.com/homedetails/225-Mystic-...,https://photos.zillowstatic.com/fp/e92a0b0b33e...,2.0,1.0,1064.0,"4,818 sqft","225 Mystic St, Arlington, MA 02474",4978.0,single family,751.0,1880.0,Arlington,granite countertops; stainless steel; fireplace,bonus room; living room; dining room,range; oven,"circuit breakers, 100 amp service",public sewer,public,"for gas range, for electric oven",3.0,"paved drive, tandem, paved",69.0,59.0,Bishop,0.1,Ottoson,0.1,Arlington,0.7,1.0,1.0,6.0,3.0,5.0,Arlington; Cambridge; Everett; Framingham; Lowell,"{'date': '6/25/2025', 'event': 'Listed for sal...",1.0,NaN,799000,Ultra Compact,4.0,2474,145.0,65.0,0.300,2.85,3.0,1.0,0,Adequate,Middlesex,Mid-tier,None,"circuit breakers, 100 amp service public sewer...",granite countertops; stainless steel; fireplac...
2,https://www.zillow.com/homedetails/43-Longmead...,https://photos.zillowstatic.com/fp/740e93bfcff...,2.0,1.0,1628.0,"7,954 sqft","43 Longmeadow Rd, Arlington, MA 02474",5603.0,single family,552.0,1952.0,Arlington,fireplace; laminate,living room; family room; dining room,range; oven; washer,circuit breakers,public sewer,public,"for gas range, for gas oven, for gas dryer, wa...",5.0,off street,32.0,35.0,Nearby,5.0,Ottoson,0.2,Additional,5.0,1.0,1.0,6.0,3.0,5.0,Arlington; Cambridge; Everett; Framingham; Lowell,"{'date': '7/22/2025', 'event': 'Listed for sal...",NaN,NaN,899000,Ultra Compact,7.0,2474,73.0,33.2,3.400,2.85,5.0,0.0,0,Abundant,Middlesex,Mid-tier,None,circuit breakers public sewer public for gas r...,fireplace; laminate living room; family room; ...
3,https://www.zillow.com/homedetails/11-Pine-Ct-...,https://photos.zillowstatic.com/fp/bb84e595f96...,3.0,2.0,1824.0,"6,037 sqft","11 Pine Ct, Arlington, MA 02476",6219.0,single family,547.0,1926.0,Arlington,fireplace; hardwood floors,living room; office; dining room,range,,public sewer,public,,2.0,off street,84.0,54.0,Brackett,0.7,Ottoson,0.7,Arlington,0.7,1.0,1.0,6.0,3.0,6.0,Arlington; Cambridge; Everett; Framingham; Lowell,"{'date': '7/22/2025', 'event': 'Listed for sal...",NaN,NaN,998000,Ultra Compact,6.0,2476,99.0,72.0,0.700,2.95,2.0,0.0,0,Basic,Middlesex,Mid-tier,None,public sewer public,fireplace; hardwood floors living room; office...
4,https://www.zillow.com/homedetails/17-Norcross...,https://photos.zillowstatic.com/fp/ac474eb496d...,2.0,2.0,1221.0,NaN,"17 Norcross St FLOOR 3, Arlington, MA 024

In [ ]:
df['price'].max()

95000000

In [ ]:
df[df['price'] == 95000000]

,url,image_url,beds,baths,sqft,sqft_lot,address,estimated_monthly_payment,property_type,price_per_sqft,year_built,region,interior_features,other_rooms,appliances,utilities_Electric,utilities_Sewer,utilities_Water,utilities_Utilities,parking_total_spaces,parking_parking_features,walk_score,bike_score,elementary_school_name,elementary_school_distance,middle_school_name,middle_school_distance,high_school_name,high_school_distance,flood_risk,fire_risk,wind_risk,air_risk,heat_risk,nearby_cities,property_history,parking_uncovered_spaces,parking_garage_spaces,price,sqft_lot_caterogy,sqft_lot_cleaned,zip_code,property_age,mobility_score,avg_school_distance_miles,weighted_risk_score,parking_spaces,parking_quality_score,has_garage,parking_adequacy,county,zip_price_tier,"(2884, utilities_Water)",combined_utilities,combined_interior_features
599,https://www.zillow.com/homedetails/90-100-Bria...,https://photos.zillowstatic.com/fp/739424bd9f8...,10.0,15.0,13800.0,11.2 Acres,"90 & 100 Briarpatch Road, East Hampton, NY 11937",687228.0,single family,6884.0,1931.0,East Hampton,walk-in closet; marble; fireplace,office; dining room; living room,range,,septic tank,public,,NaN,"detached, private",8.0,26.0,John M Marshall,2.1,East Hampton,2.1,Located south of the,2.4,NaN,NaN,NaN,NaN,NaN,Bay Shore; Brentwood; Central Islip; Huntingto...,"{'date': '11/20/2024', 'event': 'Listed for sa...",NaN,NaN,95000000,Rural/Estate,487872.0,NaN,94.0,15.2,2.2,NaN,0.0,0.0,0,Limited,Other,Mid-tier,None,septic tank public,walk-in closet; marble; fireplace office; dini...


In [ ]:
import pickle

# Define the filename for the pickle file
pickle_filename = 'property_recommender_components.pkl'

# Combine the cosine similarity matrices with the specified weights outside the function
combined_cosine_sim_matrix = (30 * cosine_sim1 +
                              20 * cosine_sim_interior +
                              8 * cosine_sim_cities)

# Create a dictionary to hold the components you want to export
recommender_components = {
    'tfidf_vectorizer': tfidf_vectorizer,
    'tfidf_matrix': tfidf_matrix,
    'tfidf_vectorizer_interior': tfidf_vectorizer_interior,
    'tfidf_matrix_interior': tfidf_matrix_interior,
    'tfidf_vectorizer_cities': tfidf_vectorizer_cities,
    'tfidf_matrix_cities': tfidf_matrix_cities,
    'combined_cosine_sim_matrix': combined_cosine_sim_matrix,
    'df_addresses': df['address'], # Include addresses for lookup
    'df_prices': df['price'] # Include prices for lookup
}

# Open the file in write-binary mode and dump the dictionary
with open(pickle_filename, 'wb') as f:
    pickle.dump(recommender_components, f)

print(f"Recommender components saved to {pickle_filename}")

Recommender components saved to property_recommender_components.pkl
